In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Ravi2022_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["P234_JAK", "P294", "P317", "P322", "X154", "X192", "X194"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 40436 × 17916
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [8]:
#adata = adata.raw.to_adata()

In [9]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 436/436 [00:00<00:00, 690.17it/s]


In [10]:
adata.X = X_counts_recovered

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [12]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [13]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [14]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [15]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF367,False,2590,6.405183,True,0.030726,0.019751,0.656884
SULT1B1,False,471,1.164804,True,0.005379,0.003267,0.653755
TRIM63,False,123,0.304184,True,0.001179,0.000639,0.505263
HDHD2,False,3677,9.093382,True,0.032923,0.014319,0.432469
MORF4L2-AS1,False,789,1.951232,True,0.008830,0.005348,0.639297
...,...,...,...,...,...,...,...
CPHXL,False,146,0.361064,True,0.001426,0.000816,0.612434
LINC01643,False,63,0.155802,True,0.000683,0.000370,0.567259
NCF4-AS1,False,208,0.514393,True,0.002447,0.001470,0.640853


In [16]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [17]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [18]:
adata.var = df_tmp

In [19]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [20]:
adata

View of AnnData object with n_obs × n_vars = 40436 × 16565
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [21]:
adata.obs['donor_id'] = df_obs['donor_id']

In [22]:
# Create a DataFrame with the metadata for Ravi2022
metadata_data = {
    'Author': ['Ravi2022'] * 7,
    'donor_id': ["P234_JAK", "P294", "P317", "P322", "X154", "X192", "X194"],
    'stage': ['Primary'] * 7,
    'assay': ['10x 3\' v3'] * 7,
    'tissue': ['parietal lobe', 'temporal lobe', 'temporal lobe', 'frontal lobe', 'parietal lobe', 'parietal lobe', 'temporal lobe'],
    'Cells': ['Total'] * 7,
    'Method': ['cell'] * 7
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

     Author  donor_id    stage      assay         tissue  Cells Method
0  Ravi2022  P234_JAK  Primary  10x 3' v3  parietal lobe  Total   cell
1  Ravi2022      P294  Primary  10x 3' v3  temporal lobe  Total   cell
2  Ravi2022      P317  Primary  10x 3' v3  temporal lobe  Total   cell
3  Ravi2022      P322  Primary  10x 3' v3   frontal lobe  Total   cell
4  Ravi2022      X154  Primary  10x 3' v3  parietal lobe  Total   cell
5  Ravi2022      X192  Primary  10x 3' v3  parietal lobe  Total   cell
6  Ravi2022      X194  Primary  10x 3' v3  temporal lobe  Total   cell


In [23]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

       donor_id    Author    stage      assay         tissue  Cells Method
0      P234_JAK  Ravi2022  Primary  10x 3' v3  parietal lobe  Total   cell
1      P234_JAK  Ravi2022  Primary  10x 3' v3  parietal lobe  Total   cell
2      P234_JAK  Ravi2022  Primary  10x 3' v3  parietal lobe  Total   cell
3      P234_JAK  Ravi2022  Primary  10x 3' v3  parietal lobe  Total   cell
4      P234_JAK  Ravi2022  Primary  10x 3' v3  parietal lobe  Total   cell
...         ...       ...      ...        ...            ...    ...    ...
40431      X194  Ravi2022  Primary  10x 3' v3  temporal lobe  Total   cell
40432      X194  Ravi2022  Primary  10x 3' v3  temporal lobe  Total   cell
40433      X194  Ravi2022  Primary  10x 3' v3  temporal lobe  Total   cell
40434      X194  Ravi2022  Primary  10x 3' v3  temporal lobe  Total   cell
40435      X194  Ravi2022  Primary  10x 3' v3  temporal lobe  Total   cell

[40436 rows x 7 columns]


In [24]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [25]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
Ravi2022_AAACCCAAGGTGCGAT-1_2-0-1,P234_JAK,3367,2315.467285,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
Ravi2022_AAACCCAAGTGGTCAG-1_2-0-1,P234_JAK,3228,2344.219971,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
Ravi2022_AAACCCACAGTTAGGG-1_2-0-1,P234_JAK,5118,2558.255859,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
Ravi2022_AAACCCACATGACGAG-1_2-0-1,P234_JAK,2254,2232.778320,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
Ravi2022_AAACCCAGTTCGGACC-1_2-0-1,P234_JAK,2927,2160.298584,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Microglia,microglial cell
...,...,...,...,...,...,...,...,...,...
Ravi2022_TTTGTTGGTGTGTCGC-1_2_1_1-0-1,X194,1547,1899.195557,Non-neoplastic,Lymphoid,B cell,B cell,B Cells,B cell
Ravi2022_TTTGTTGGTTTGCAGT-1_2_1_1-0-1,X194,2696,2322.857422,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
Ravi2022_TTTGTTGTCACTACTT-1_2_1_1-0-1,X194,5142,2624.053223,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
Ravi2022_TTTGTTGTCGCGCTGA-1_2_1_1-0-1,X194,1823,1990.109863,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Microglia,macrophage


In [26]:
merged_obs_df.index= df_obs.index

In [27]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [28]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [29]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [30]:
del merged_obs_df['donor_id_y']

In [31]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [32]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [33]:
adata.obs = merged_obs_df

In [34]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    729 total control genes are used. (0:00:02)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    727 total control genes are used. (0:00:02)
-->     'phase', cell cycle phase (adata.obs)


In [35]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Ravi2022_Part3.h5ad")